[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week07/notebooks/AG952_Week07_Activity1_SentimentDictionaries.ipynb)

## AG952 | Week 7 | Activity 1: Sentiment Dictionaries in Financial Text

### Comparing the Harvard General Inquirer against Loughran and McDonald (2011) on a real SEC 10-K filing

---

The aim of this activity is to show how dictionary choice affects the measured sentiment of a real corporate filing. Loughran and McDonald (2011) demonstrate that the Harvard General Inquirer (GI) treats a large share of standard accounting terminology -- words such as *cost*, *tax*, *liability*, and *risk* -- as negative, even though these terms carry no adverse signal in a financial context. The result is that Harvard-based sentiment scores are systematically inflated, which can corrupt empirical tests of the relationship between textual sentiment and financial outcomes. By running both dictionaries on the same filing, you will see the magnitude of this problem in concrete terms.

**How to use this notebook:** run each cell in order using Shift+Enter. Each cell is labelled with its purpose. Read the brief notes between cells before running the next step.

**Estimated time:** 20 minutes.

In [ ]:
#@title Step 1 -- Install and import
!pip install requests beautifulsoup4 matplotlib seaborn wordcloud pandas -q

import re
import time
import warnings
from collections import Counter

import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from wordcloud import WordCloud
import pandas as pd

warnings.filterwarnings("ignore")

print("Step 1 complete -- all libraries imported.")


In [ ]:
#@title Step 2 -- Load word lists

# Harvard General Inquirer H4N negative list -- words commonly found in 10-Ks
# but which are financially neutral. These are the false positives.
HARVARD_NEGATIVE = {
    "tax", "taxes", "cost", "costs", "capital", "capitalize", "capitalization",
    "expense", "expenses", "liability", "liabilities", "depreciation",
    "service", "services", "risk", "risks", "board", "foreign", "lower",
    "limited", "decrease", "decreased", "loss", "losses", "claim", "claims",
    "damage", "damages", "adverse", "adversely", "decline", "default",
    "failure", "fraud", "impairment", "litigation", "penalty", "penalties",
    "problem", "problems", "restructuring", "terminated", "termination",
    "unable", "volatile", "volatility", "weakness", "weaknesses", "deficit",
    "poor", "error", "errors", "difficult", "difficulty", "difficulties",
    "insufficient", "breach", "concern", "threat", "negative", "negatively",
    "corrupt", "corruption", "lawsuit", "bankrupt", "bankruptcy",
    "writedown", "writeoff", "shortage", "shortfall", "investigation",
    "burden", "unfavorable", "controversy", "conflict", "critical",
    "crisis", "cease", "ceased", "doubtful", "dispute", "disputed",
    "fine", "fines", "neglect", "negligence", "noncompliance",
    "layoff", "layoffs", "misconduct", "insolvency", "insolvent",
    "delinquent", "delinquency", "downgrade", "downgraded", "downturn",
}

# Loughran and McDonald (2011) -- words with genuine adverse financial meaning
LM_NEGATIVE = {
    "abandon", "abandoned", "abandonment", "adverse", "adversely", "adversity",
    "allegation", "allegations", "breach", "breached", "burden", "cease",
    "ceased", "claim", "claims", "complaint", "complaints", "conflict",
    "corrupt", "corruption", "crisis", "critical", "damage", "damages",
    "decline", "declined", "declines", "declining", "default", "defaulted",
    "defaults", "deficit", "deficits", "difficult", "difficulties", "difficulty",
    "discontinued", "dispute", "disputed", "doubtful", "error", "errors",
    "fail", "failed", "failure", "failures", "false", "fine", "fines",
    "fraud", "fraudulent", "harm", "harmful", "impair", "impaired",
    "impairment", "impairments", "inadequate", "inability", "insufficient",
    "lawsuit", "lawsuits", "layoff", "layoffs", "litigation", "loss", "losses",
    "negative", "negatively", "neglect", "negligence", "noncompliance",
    "penalty", "penalties", "poor", "poorly", "problem", "problems",
    "restructuring", "shortage", "shortfall", "terminated", "termination",
    "unable", "volatile", "volatility", "vulnerability", "weak", "weakness",
    "weaknesses", "writedown", "writeoff", "bankrupt", "bankruptcy",
    "downgrade", "downgraded", "downturn", "restatement", "restated",
    "investigation", "misconduct", "insolvency", "insolvent",
    "delinquent", "delinquency",
}

# Harvard false positives: in Harvard but not in L&M
HARVARD_FALSE_POSITIVES = HARVARD_NEGATIVE - LM_NEGATIVE

print(f"Harvard GI negative list size : {len(HARVARD_NEGATIVE):,}")
print(f"L&M negative list size        : {len(LM_NEGATIVE):,}")
print(f"Harvard false positives       : {len(HARVARD_FALSE_POSITIVES):,}")
print()
print("Example Harvard false positives (in Harvard but not L&M):")
fp_examples = sorted(HARVARD_FALSE_POSITIVES)[:8]
for w in fp_examples:
    print(f"  {w}")

print("\nStep 2 complete -- word lists loaded.")


In [ ]:
#@title Step 3 -- Download Apple's 10-K from EDGAR

HEADERS = {"User-Agent": "AG952 Strathclyde University ag952course@strath.ac.uk"}
EDGAR_BASE = "https://data.sec.gov"
ARCHIVES_BASE = "https://www.sec.gov/Archives/edgar/data"


def get_latest_10k(cik):
    """Fetch metadata for the most recent 10-K filing for a given CIK."""
    cik_padded = str(cik).zfill(10)
    url = f"{EDGAR_BASE}/submissions/CIK{cik_padded}.json"
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
        time.sleep(0.2)
        company = data.get("name", "Unknown")
        recent = data["filings"]["recent"]
        for i, form in enumerate(recent["form"]):
            if form == "10-K":
                return {
                    "company": company,
                    "cik": str(cik),
                    "date": recent["filingDate"][i],
                    "accession": recent["accessionNumber"][i],
                    "primary_doc": recent["primaryDocument"][i],
                }
    except Exception as e:
        print(f"Error fetching metadata: {e}")
    return None


def download_10k_text(filing, max_chars=600000):
    """Download and clean the primary document of a 10-K filing."""
    try:
        acc_clean = filing["accession"].replace("-", "")
        cik_str = str(int(filing["cik"]))
        url = f"{ARCHIVES_BASE}/{cik_str}/{acc_clean}/{filing['primary_doc']}"
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        time.sleep(0.2)
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["table", "script", "style", "ix:header", "ix:nonfraction"]):
            tag.decompose()
        text = soup.get_text(separator=" ")
        text = re.sub(r"\s+", " ", text).strip()
        return text[:max_chars]
    except Exception as e:
        print(f"Error downloading filing: {e}")
        return ""


print("Fetching Apple Inc. 10-K from SEC EDGAR...")
apple_filing = get_latest_10k(320193)

if apple_filing:
    print(f"Company     : {apple_filing['company']}")
    print(f"Filing date : {apple_filing['date']}")
    print(f"Accession   : {apple_filing['accession']}")
    print("\nDownloading filing text...")
    apple_text = download_10k_text(apple_filing)
    word_count = len(apple_text.split())
    print(f"Characters  : {len(apple_text):,}")
    print(f"Words       : {word_count:,}")
else:
    print("Could not retrieve filing metadata from EDGAR.")
    apple_text = ""

print("\nStep 3 complete -- filing downloaded.")


In [ ]:
#@title Step 4 -- Apply both dictionaries

def score_document(text, word_list):
    """Tokenise text and return matched tokens and total word count."""
    tokens = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    matched = [t for t in tokens if t in word_list]
    return matched, len(tokens)


if apple_text:
    harvard_matches, total_words = score_document(apple_text, HARVARD_NEGATIVE)
    lm_matches, _ = score_document(apple_text, LM_NEGATIVE)

    # Partition Harvard matches
    genuine_negatives = [t for t in harvard_matches if t in LM_NEGATIVE]
    false_positives   = [t for t in harvard_matches if t in HARVARD_FALSE_POSITIVES]

    harvard_score  = len(harvard_matches) / total_words * 100
    lm_score       = len(lm_matches) / total_words * 100
    overstatement  = harvard_score / lm_score if lm_score > 0 else float("nan")
    fp_pct         = len(false_positives) / len(harvard_matches) * 100 if harvard_matches else 0

    # Frequency counters for visualisation
    fp_freq  = Counter(false_positives)
    gen_freq = Counter(genuine_negatives)

    print("=" * 60)
    print(f"Filing: {apple_filing['company']} 10-K ({apple_filing['date'][:4]})")
    print("=" * 60)
    print(f"Total words in filing            : {total_words:>10,}")
    print(f"Harvard GI negative matches      : {len(harvard_matches):>10,}")
    print(f"  of which genuine (also in L&M) : {len(genuine_negatives):>10,}")
    print(f"  of which false positives       : {len(false_positives):>10,}")
    print(f"L&M negative matches             : {len(lm_matches):>10,}")
    print(f"Harvard sentiment score          : {harvard_score:>10.3f}%")
    print(f"L&M sentiment score              : {lm_score:>10.3f}%")
    print(f"Overstatement factor (H / L&M)   : {overstatement:>10.2f}x")
    print(f"Harvard false positive rate      : {fp_pct:>10.1f}%")
    print("=" * 60)
    print()
    lm_ref = 73.8
    print(f"Loughran and McDonald (2011) report that up to {lm_ref}% of Harvard negatives")
    print(f"in 10-Ks are false positives. This filing shows {fp_pct:.1f}%.")
else:
    print("No filing text available -- please re-run Step 3.")

print("\nStep 4 complete -- dictionaries applied.")


In [ ]:
#@title Step 5 -- Visualise the misclassification

if apple_text and apple_filing:
    company_name = apple_filing["company"]
    year_str     = apple_filing["date"][:4]

    FP_COL  = "#f5a623"
    GEN_COL = "#e94560"
    TXT_COL = "#e0e0e0"
    GRID_COL= "#2a2a4a"
    FIG_BG  = "#0f0f1a"
    AX_BG   = "#1a1a2e"

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor(FIG_BG)

    for ax in axes:
        ax.set_facecolor(AX_BG)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(colors=TXT_COL)
        ax.xaxis.label.set_color(TXT_COL)
        ax.yaxis.label.set_color(TXT_COL)
        ax.title.set_color(TXT_COL)

    # Panel 1: Top 12 Harvard false positives
    top_fp = fp_freq.most_common(12)
    top_fp_sorted = sorted(top_fp, key=lambda x: x[1])
    fp_labels, fp_vals = zip(*top_fp_sorted)
    bars1 = axes[0].barh(range(len(fp_labels)), fp_vals, color=FP_COL, height=0.65)
    axes[0].set_yticks(range(len(fp_labels)))
    axes[0].set_yticklabels(fp_labels, fontsize=8.5, color=TXT_COL)
    axes[0].set_xlabel("Frequency", fontsize=9, color=TXT_COL)
    axes[0].set_title("Top Harvard 'Negative' Words\nThat Are Neutral in Finance",
                      fontsize=9.5, fontweight="bold", color=TXT_COL)
    axes[0].grid(axis="x", color=GRID_COL, linewidth=0.5)
    for bar, val in zip(bars1, fp_vals):
        axes[0].text(val + max(fp_vals)*0.01, bar.get_y() + bar.get_height()/2,
                     str(val), va="center", fontsize=7.5, color=TXT_COL)

    # Panel 2: Top 12 genuine negatives
    top_gen = gen_freq.most_common(12)
    top_gen_sorted = sorted(top_gen, key=lambda x: x[1])
    gen_labels, gen_vals = zip(*top_gen_sorted)
    bars2 = axes[1].barh(range(len(gen_labels)), gen_vals, color=GEN_COL, height=0.65)
    axes[1].set_yticks(range(len(gen_labels)))
    axes[1].set_yticklabels(gen_labels, fontsize=8.5, color=TXT_COL)
    axes[1].set_xlabel("Frequency", fontsize=9, color=TXT_COL)
    axes[1].set_title("Genuine Negative Words\n(Flagged by Both Dictionaries)",
                      fontsize=9.5, fontweight="bold", color=TXT_COL)
    axes[1].grid(axis="x", color=GRID_COL, linewidth=0.5)
    for bar, val in zip(bars2, gen_vals):
        axes[1].text(val + max(gen_vals)*0.01, bar.get_y() + bar.get_height()/2,
                     str(val), va="center", fontsize=7.5, color=TXT_COL)

    # Panel 3: Score comparison
    bar_labels = ["Harvard GI", "L&M (2011)"]
    bar_vals   = [harvard_score, lm_score]
    bar_cols   = [FP_COL, GEN_COL]
    bars3 = axes[2].bar(bar_labels, bar_vals, color=bar_cols, width=0.45)
    axes[2].set_ylabel("% of total words", fontsize=9, color=TXT_COL)
    axes[2].set_title("Negative Sentiment Score\nComparison",
                      fontsize=9.5, fontweight="bold", color=TXT_COL)
    axes[2].yaxis.grid(True, linestyle=":", color=GRID_COL, linewidth=0.7)
    for bar, val in zip(bars3, bar_vals):
        axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                     f"{val:.3f}%", ha="center", va="bottom",
                     fontsize=9, color=TXT_COL, fontweight="bold")
    arrow_x = 0  # Harvard bar index
    axes[2].annotate(
        f"Harvard overstates by {overstatement:.1f}x",
        xy=(0, harvard_score), xytext=(0.55, harvard_score * 0.75),
        fontsize=8.5, color=FP_COL,
        arrowprops=dict(arrowstyle="->", color=FP_COL, lw=1.2),
        color=FP_COL
    )

    fig.suptitle(
        f"Sentiment Dictionary Comparison: {company_name} 10-K ({year_str}) -- "
        f"Harvard GI vs. Loughran and McDonald (2011)",
        fontsize=11, fontweight="bold", color=TXT_COL, y=1.02
    )
    plt.tight_layout()
    plt.savefig("sentiment_dictionary_comparison.png", dpi=130, bbox_inches="tight",
                facecolor=FIG_BG)
    plt.show()
else:
    print("No data available -- please re-run Steps 3 and 4 first.")

print("\nStep 5 complete -- visualisation saved.")


In [ ]:
#@title Step 6 -- Word frequency clouds

if apple_text and apple_filing:
    company_name = apple_filing["company"]
    FIG_BG = "#0f0f1a"
    CLOUD_BG = "#1a1a2e"

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor(FIG_BG)

    # Left: Harvard false positives
    if fp_freq:
        wc_fp = WordCloud(
            background_color=CLOUD_BG, max_words=60,
            prefer_horizontal=0.9, min_font_size=10,
            color_func=lambda *args, **kwargs: "#f5a623"
        ).generate_from_frequencies(fp_freq)
        axes[0].imshow(wc_fp, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_facecolor(CLOUD_BG)
    axes[0].set_title(
        "Harvard flags these as negative --\nL&M classifies them as neutral in a financial context",
        fontsize=9.5, color="white", fontweight="bold"
    )

    # Right: Genuine negatives
    if gen_freq:
        wc_gen = WordCloud(
            background_color=CLOUD_BG, max_words=60,
            prefer_horizontal=0.9, min_font_size=10,
            color_func=lambda *args, **kwargs: "#e94560"
        ).generate_from_frequencies(gen_freq)
        axes[1].imshow(wc_gen, interpolation="bilinear")
    axes[1].axis("off")
    axes[1].set_facecolor(CLOUD_BG)
    axes[1].set_title(
        "Words classified as genuinely negative\nby both dictionaries",
        fontsize=9.5, color="white", fontweight="bold"
    )

    fig.suptitle(
        f"{company_name} 10-K -- False positives (orange) vs. genuine negative signals (red)",
        fontsize=11, fontweight="bold", color="white", y=1.02
    )
    plt.tight_layout()
    plt.savefig("sentiment_wordclouds.png", dpi=130, bbox_inches="tight",
                facecolor=FIG_BG)
    plt.show()
else:
    print("No data available -- please re-run earlier steps first.")

print("\nStep 6 complete -- word clouds saved.")


## Key Takeaways

1. **The Harvard GI treats standard accounting terminology as negativity.** Words such as *cost*, *tax*, *liability*, and *risk* appear throughout every 10-K as a matter of routine disclosure, not because a firm is in distress. The Harvard list was constructed for general English text and has no way to distinguish accounting vocabulary from emotionally negative language.

2. **The L&M (2011) dictionary excludes accounting terminology.** Loughran and McDonald constructed their list by identifying words that are negative specifically in a 10-K context -- words that appear disproportionately in filings associated with poor financial outcomes. The result is a much lower but more precise signal.

3. **The overstatement is large enough to corrupt regression results.** When Harvard sentiment scores are used to predict stock returns or earnings, any firm with complex operations generates a high negative score regardless of its performance. This introduces systematic measurement error that can bias coefficient estimates and inflate statistical significance.

4. **This is not a minor calibration issue.** Loughran and McDonald (2011) show that up to 73.8% of Harvard negatives in 10-Ks are false positives. Dictionary choice can alter the sign and significance of published findings, not just shift them slightly.

---

> **Discussion question:** If a researcher uses the Harvard GI to study whether sentiment predicts earnings, which type of firm will always appear pessimistic regardless of its actual performance? What are the consequences for their regression coefficients?

---

### Further Reading

Loughran, T. and McDonald, B. (2011) 'When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks', *Journal of Finance*, 66(1), pp. 35--65.

Full L&M master dictionary: https://sraf.nd.edu/loughranmcdonald-master-dictionary/